Note: Pandas ka merge() SQL ke JOIN jaisa hi hai (jo hum Month 1 mein detail se kar chuke ho — INNER, LEFT, RIGHT, FULL OUTER). Concept same hai, syntax Pandas ka hai.

1. Do chhote DataFrames banao practice ke liye:

In [1]:
import pandas as pd

customers = pd.DataFrame({
    'Customer ID': [12346, 12347, 12348, 12349, 12350],
    'Country': ['United Kingdom', 'Germany', 'France', 'Spain', 'Italy']
})

churn_data = pd.DataFrame({
    'Customer ID': [12346, 12347, 12348, 99999],   # 99999 customers mein nahi hai (demo ke liye)
    'Churned': [1, 0, 1, 0]
})

print(customers)
print(churn_data)

   Customer ID         Country
0        12346  United Kingdom
1        12347         Germany
2        12348          France
3        12349           Spain
4        12350           Italy
   Customer ID  Churned
0        12346        1
1        12347        0
2        12348        1
3        99999        0


2. INNER JOIN (default) — sirf dono mein match wale rows:

In [2]:
inner_result = pd.merge(customers, churn_data, on='Customer ID', how='inner')
print(inner_result)

   Customer ID         Country  Churned
0        12346  United Kingdom        1
1        12347         Germany        0
2        12348          France        1


- Notice : 12349, 12350 (churn_data mein nahi the) aur 99999 (customers mein nahi tha) — sab gayab ho gaye, sirf match wale (3 rows) bache.

3. LEFT JOIN — left table (customers) ke sab rows, right se match wala data:

In [3]:
left_result = pd.merge(customers, churn_data, on='Customer ID', how='left')
print(left_result)

   Customer ID         Country  Churned
0        12346  United Kingdom      1.0
1        12347         Germany      0.0
2        12348          France      1.0
3        12349           Spain      NaN
4        12350           Italy      NaN


Yahan sab 5 customers dikhenge, lekin jinke liye churn_data mein match nahi tha (12349, 12350), unka Churned NaN aayega.

4. RIGHT JOIN — right table (churn_data) ke sab rows:

In [4]:
right_result = pd.merge(customers, churn_data, on='Customer ID', how='right')
print(right_result)

   Customer ID         Country  Churned
0        12346  United Kingdom        1
1        12347         Germany        0
2        12348          France        1
3        99999             NaN        0


Yahan 99999 bhi dikhega (churn_data mein tha), lekin uska Country NaN hoga.

5. OUTER JOIN — dono taraf ke sab rows (FULL OUTER JOIN jaisa, Day 8 yaad hai):

In [5]:
outer_result = pd.merge(customers, churn_data, on='Customer ID', how='outer')
print(outer_result)

   Customer ID         Country  Churned
0        12346  United Kingdom      1.0
1        12347         Germany      0.0
2        12348          France      1.0
3        12349           Spain      NaN
4        12350           Italy      NaN
5        99999             NaN      0.0


6. Apne real data par apply karna — transactions ko churn_labeled data se merge karna:

In [6]:
df = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str})
churn_labels = pd.read_csv('../data/processed/customer_churn_labeled.csv')

merged = pd.merge(df, churn_labels[['Customer ID', 'Churned']], on='Customer ID', how='left')
print(merged.shape)
print(merged[['Customer ID', 'Country', 'Churned']].head(10))

(779495, 14)
   Customer ID         Country  Churned
0        13085  United Kingdom        1
1        13085  United Kingdom        1
2        13085  United Kingdom        1
3        13085  United Kingdom        1
4        13085  United Kingdom        1
5        13085  United Kingdom        1
6        13085  United Kingdom        1
7        13085  United Kingdom        1
8        13085  United Kingdom        1
9        13085  United Kingdom        1


7. Different column names hone par merge karna (left_on, right_on):

In [7]:
churn_labels_renamed = churn_labels.rename(columns={'Customer ID': 'CustID'})

merged2 = pd.merge(df, churn_labels_renamed, left_on='Customer ID', right_on='CustID', how='left')
print(merged2.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Year', 'Month', 'Day', 'DayOfWeek', 'Hour', 'CustID', 'Frequency', 'Monetary', 'Recency', 'Churned']


8. .join() — index-based merging (shortcut jab index common ho):

In [8]:
customers_indexed = customers.set_index('Customer ID')
churn_indexed = churn_data.set_index('Customer ID')

joined = customers_indexed.join(churn_indexed, how='left')
print(joined)

                    Country  Churned
Customer ID                         
12346        United Kingdom      1.0
12347               Germany      0.0
12348                France      1.0
12349                 Spain      NaN
12350                 Italy      NaN


- Note: .join() .merge() ka hi ek chhota version hai, specifically index par based — jab dono tables ka index hi common key ho to .join() likhna zyada concise hai.

9. Merge validation — duplicate keys se accidental row explosion check karna (important gotcha):

In [9]:
# validate parameter se pata chal jaata hai agar merge unexpectedly rows badha raha hai
try:
    check = pd.merge(customers, churn_data, on='Customer ID', how='left', validate='one_to_one')
    print("Merge is clean one-to-one")
except Exception as e:
    print(f"Validation issue: {e}")

Merge is clean one-to-one


Practice questions:

1. Ek naya chhota DataFrame banao "country_region" mapping ka (jaise {'United Kingdom': 'Europe', 'Germany': 'Europe', 'India': 'Asia'}), aur apne main df ke saath LEFT JOIN karo Country par.

In [12]:
import pandas as pd

# Main dataframe ko dobara load karo (agar loaded nahi hai)
df = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str})

# 1. Country-Region mapping DataFrame banana
region_mapping = pd.DataFrame({
    'Country': ['United Kingdom', 'Germany', 'France', 'Spain', 'Italy', 'EIRE', 'Portugal'],
    'Region': ['Europe', 'Europe', 'Europe', 'Europe', 'Europe', 'Europe', 'Europe']
})

# 2. Left join karna 'Country' ke base par
df_with_region = pd.merge(df, region_mapping, on='Country', how='left')

print("--- Merged DataFrame with Region ---")
print(df_with_region[['Invoice', 'Country', 'Region']].head(10))
print(f"Total rows after merge: {df_with_region.shape[0]}")

--- Merged DataFrame with Region ---
  Invoice         Country  Region
0  489434  United Kingdom  Europe
1  489434  United Kingdom  Europe
2  489434  United Kingdom  Europe
3  489434  United Kingdom  Europe
4  489434  United Kingdom  Europe
5  489434  United Kingdom  Europe
6  489434  United Kingdom  Europe
7  489434  United Kingdom  Europe
8  489435  United Kingdom  Europe
9  489435  United Kingdom  Europe
Total rows after merge: 779495


2. merge() se pehle aur baad mein .shape[0] compare karna practice karo — agar row count unexpectedly badh jaye, samajh jao ki duplicate keys hain kisi table mein (jaise SQL mein CROSS JOIN accidentally ho jaana).

In [13]:
churn_labels = pd.read_csv('../data/processed/customer_churn_labeled.csv')

# Merge se pehle main dataframe ki row count
rows_before = df.shape[0]
print(f"Rows before merge: {rows_before}")

# Merge perform karo
merged_df = pd.merge(df, churn_labels[['Customer ID', 'Churned']], on='Customer ID', how='left')

# Merge ke baad ki row count
rows_after = merged_df.shape[0]
print(f"Rows after merge: {rows_after}")

# Comparison check
if rows_before == rows_after:
    print(" Safe: Row count match ho rahi hai, koi row explosion nahi hua!")
else:
    print(" Warning: Rows badh gayi hain! Check for duplicate keys in the right table.")

Rows before merge: 779495
Rows after merge: 779495
 Safe: Row count match ho rahi hai, koi row explosion nahi hua!
